# Día 10 · Unidad 12 — Cálculo de provisiones: método Chain Ladder

**Objetivos de aprendizaje**
- Representar un triángulo de siniestralidad acumulada como `DataFrame` de pandas.
- Calcular los factores de desarrollo (*link ratios*) y proyectar el triángulo hasta "última pérdida" (*ultimate*).
- Calcular la provisión (IBNR + IBNER, en el sentido amplio de "reserva pendiente") por año de origen y en total.
- Validar la proyección con dos casos de control: factores de desarrollo iguales a 1 (sin desarrollo pendiente) y un triángulo con un patrón de desarrollo constante conocido de antemano.

**Duración estimada:** 45-60 minutos.

**Requisitos previos:** `06_teoria_copulas.ipynb` de este mismo día. Se asume dominio previo del método Chain Ladder — aquí el foco es exclusivamente la implementación en pandas/NumPy.

## 1. El triángulo de siniestralidad acumulada

Construimos a mano un triángulo pequeño (5 años de origen x 5 años de desarrollo) de siniestros pagados **acumulados**: la fila es el año de origen (accidente/emisión), la columna es el año de desarrollo (0 = primer año, 1 = segundo año...), y cada celda es el importe acumulado pagado hasta ese punto de desarrollo. Las celdas por debajo de la diagonal (futuro aún no observado) son `NaN`.

In [1]:
import numpy as np
import pandas as pd

anios_origen = [2020, 2021, 2022, 2023, 2024]
periodos_desarrollo = [0, 1, 2, 3, 4]

# Triangulo de pagos acumulados (en miles de EUR), inventado a mano para el ejemplo.
# Cada fila tiene menos observaciones cuanto mas reciente es el anio de origen
# (2024 solo tiene el primer periodo de desarrollo observado).
datos_triangulo = [
    [1_200, 1_950, 2_150, 2_230, 2_260],   # 2020: totalmente desarrollado
    [1_300, 2_070, 2_290, 2_360, np.nan],  # 2021: falta el periodo 4
    [1_400, 2_220, 2_460, np.nan, np.nan], # 2022: faltan periodos 3 y 4
    [1_500, 2_380, np.nan, np.nan, np.nan],# 2023: falta periodos 2, 3 y 4
    [1_650, np.nan, np.nan, np.nan, np.nan],  # 2024: solo periodo 0
]

triangulo = pd.DataFrame(datos_triangulo, index=anios_origen, columns=periodos_desarrollo)
triangulo.index.name = "anio_origen"
triangulo.columns.name = "periodo_desarrollo"
triangulo

periodo_desarrollo,0,1,2,3,4
anio_origen,,,,,
2020,1200,1950.0,2150.0,2230.0,2260.0
2021,1300,2070.0,2290.0,2360.0,NaN
2022,1400,2220.0,2460.0,NaN,NaN
2023,1500,2380.0,NaN,NaN,NaN
2024,1650,NaN,NaN,NaN,NaN


## 2. Factores de desarrollo (*link ratios*)

El factor de desarrollo del periodo $k$ al $k+1$ es la suma de todos los valores en el periodo $k+1$ dividida por la suma de esos mismos años de origen en el periodo $k$ — usando **solo los años de origen que tienen dato observado en ambos periodos** (el triángulo, no el rectángulo completo).

In [2]:
def calcular_factores_desarrollo(triangulo: pd.DataFrame) -> pd.Series:
    """Calcula los factores de desarrollo (link ratios) volumen-ponderados de un triangulo.

    Args:
        triangulo: DataFrame con anios de origen en el indice y periodos de
            desarrollo en las columnas (enteros consecutivos empezando en 0),
            con NaN en las celdas no observadas (futuro).

    Returns:
        pd.Series: factor de desarrollo de cada periodo k a k+1, indexado por k.
    """
    factores = {}
    columnas = sorted(triangulo.columns)
    for k, k_siguiente in zip(columnas[:-1], columnas[1:]):
        # anios de origen con dato observado en AMBOS periodos (k y k_siguiente)
        disponibles = triangulo[[k, k_siguiente]].dropna().index
        suma_k = triangulo.loc[disponibles, k].sum()
        suma_k_siguiente = triangulo.loc[disponibles, k_siguiente].sum()
        factores[k] = suma_k_siguiente / suma_k
    return pd.Series(factores, name="factor_desarrollo")


factores = calcular_factores_desarrollo(triangulo)
factores

0    1.596296
1    1.105769
2    1.033784
3    1.013453
Name: factor_desarrollo, dtype: float64

> ⚠️ **Error típico**: calcular el factor de desarrollo como la **media simple** de los ratios individuales de cada año de origen (`(ratio_2020 + ratio_2021 + ...) / n`) en vez del ratio **volumen-ponderado** (suma de todos los `k_siguiente` dividida por la suma de todos los `k`, como arriba). Ambos métodos existen en la literatura y dan resultados distintos — el volumen-ponderado (el que implementamos aquí) es el estándar Chain Ladder clásico, porque no deja que un año de origen pequeño (poco volumen) tenga el mismo peso que uno grande.

## 3. Proyección del triángulo hasta *ultimate*

Rellenamos cada celda futura (`NaN`) multiplicando la última celda observada de esa fila por los factores de desarrollo sucesivos, hasta completar el rectángulo.

In [3]:
def proyectar_triangulo(triangulo: pd.DataFrame, factores: pd.Series) -> pd.DataFrame:
    """Completa un triangulo de siniestralidad acumulada hasta el rectangulo (ultimate).

    Args:
        triangulo: triangulo de pagos acumulados, con NaN en las celdas futuras.
        factores: factores de desarrollo por periodo k -> k+1, como los que
            devuelve `calcular_factores_desarrollo`.

    Returns:
        pd.DataFrame: mismo triangulo, con las celdas futuras proyectadas
        (los valores ya observados no se modifican).
    """
    proyectado = triangulo.copy()
    columnas = sorted(proyectado.columns)
    for columna_actual, columna_siguiente in zip(columnas[:-1], columnas[1:]):
        celdas_a_proyectar = proyectado[columna_siguiente].isna()
        proyectado.loc[celdas_a_proyectar, columna_siguiente] = (
            proyectado.loc[celdas_a_proyectar, columna_actual] * factores[columna_actual]
        )
    return proyectado


triangulo_proyectado = proyectar_triangulo(triangulo, factores)
triangulo_proyectado.round(1)

periodo_desarrollo,0,1,2,3,4
anio_origen,,,,,
2020,1200,1950.0,2150.0,2230.0,2260.0
2021,1300,2070.0,2290.0,2360.0,2391.7
2022,1400,2220.0,2460.0,2543.1,2577.3
2023,1500,2380.0,2631.7,2720.6,2757.2
2024,1650,2633.9,2912.5,3010.9,3051.4


## 4. Provisión por año de origen y total

La **última pérdida estimada** (*ultimate*) de cada año de origen es la última columna del triángulo proyectado. La **provisión** (reserva pendiente de pago) es la diferencia entre esa última pérdida y lo ya pagado hasta hoy (la última celda **observada** de cada fila, es decir, la diagonal del triángulo original).

In [4]:
def calcular_provision(triangulo_original: pd.DataFrame, triangulo_proyectado: pd.DataFrame) -> pd.DataFrame:
    """Calcula la provision (reserva pendiente) por anio de origen.

    Args:
        triangulo_original: triangulo de pagos acumulados observados (con NaN
            en las celdas futuras).
        triangulo_proyectado: mismo triangulo, ya completado hasta ultimate
            (ver `proyectar_triangulo`).

    Returns:
        pd.DataFrame: una fila por anio de origen, con el pagado hasta hoy,
        la ultima perdida estimada (ultimate) y la provision de cada uno.
    """
    ultima_columna = max(triangulo_original.columns)
    pagado_hasta_hoy = triangulo_original.apply(lambda fila: fila.dropna().iloc[-1], axis=1)
    ultimate = triangulo_proyectado[ultima_columna]
    provision = ultimate - pagado_hasta_hoy
    return pd.DataFrame({
        "pagado_hasta_hoy": pagado_hasta_hoy,
        "ultimate": ultimate,
        "provision": provision,
    })


resultado_provision = calcular_provision(triangulo, triangulo_proyectado)
resultado_provision.round(1)

,pagado_hasta_hoy,ultimate,provision
anio_origen,,,
2020,2260.0,2260.0,0.0
2021,2360.0,2391.7,31.7
2022,2460.0,2577.3,117.3
2023,2380.0,2757.2,377.2
2024,1650.0,3051.4,1401.4


In [5]:
provision_total = resultado_provision["provision"].sum()
print(f"Provision total a constituir: {provision_total:,.1f} miles de EUR")

Provision total a constituir: 1,927.7 miles de EUR


## 5. Validación con dos casos de control

### Caso de control 1: año totalmente desarrollado

El año 2020 ya está completamente observado (no tiene ningún `NaN` en el triángulo original) — su provisión **debe** dar exactamente 0, porque no hay nada pendiente de desarrollar. Es una comprobación casi trivial, pero justo por eso es la primera que hay que hacer: si falla, hay un error de índices en el código, no un matiz del método.

In [6]:
provision_2020 = resultado_provision.loc[2020, "provision"]
print(f"Provision del anio 2020 (totalmente desarrollado): {provision_2020:.6f}")
assert np.isclose(provision_2020, 0.0, atol=1e-8)
print("Correcto: un anio de origen sin celdas futuras tiene provision cero.")

Provision del anio 2020 (totalmente desarrollado): 0.000000
Correcto: un anio de origen sin celdas futuras tiene provision cero.


### Caso de control 2: triángulo con desarrollo constante conocido

Construimos un triángulo sintético donde el patrón de desarrollo es exactamente conocido de antemano (cada periodo crece un 20% sobre el anterior, para todos los años de origen), y comprobamos que Chain Ladder recupera ese patrón y proyecta el resultado exacto — sin depender del triángulo "real" de las secciones anteriores.

In [7]:
factor_conocido = 1.20  # cada periodo de desarrollo es un 20% mas que el anterior
pagos_definitivos = {
    2020: [1000, 1200, 1440, 1728, 2073.6],
    2021: [1100, 1320, 1584, 1900.8, 2280.96],
    2022: [1050, 1260, 1512, 1814.4, 2177.28],
    2023: [1200, 1440, 1728, 2073.6, 2488.32],
    2024: [1300, 1560, 1872, 2246.4, 2695.68],
}
triangulo_definitivo = pd.DataFrame(pagos_definitivos, index=periodos_desarrollo).T
triangulo_definitivo.index.name = "anio_origen"
triangulo_definitivo.columns.name = "periodo_desarrollo"

# "Recortamos" el rectangulo completo a triangulo, como en cualquier triangulo real
triangulo_control = triangulo_definitivo.copy()
for i, anio in enumerate(anios_origen):
    periodos_a_ocultar = periodos_desarrollo[len(anios_origen) - i:]
    triangulo_control.loc[anio, periodos_a_ocultar] = np.nan

factores_control = calcular_factores_desarrollo(triangulo_control)
print("Factores de desarrollo recuperados:")
print(factores_control.round(6))
assert np.allclose(factores_control.values, factor_conocido)
print(f"\nCorrecto: Chain Ladder recupera exactamente el factor conocido ({factor_conocido}) en cada periodo.")

triangulo_control_proyectado = proyectar_triangulo(triangulo_control, factores_control)
assert np.allclose(triangulo_control_proyectado.values, triangulo_definitivo.values)
print("Correcto: el triangulo proyectado coincide exactamente con los pagos definitivos conocidos.")

Factores de desarrollo recuperados:
0    1.2
1    1.2
2    1.2
3    1.2
Name: factor_desarrollo, dtype: float64

Correcto: Chain Ladder recupera exactamente el factor conocido (1.2) en cada periodo.
Correcto: el triangulo proyectado coincide exactamente con los pagos definitivos conocidos.


Con un patrón de desarrollo perfectamente regular y sin ruido, Chain Ladder reconstruye el rectángulo completo de forma exacta — es el caso de control más exigente posible, porque no deja margen para "casi coincide": o el código está bien, o el `assert` falla.

**Para ti:** en el triángulo original de la sección 1, cambia el último valor de la fila 2021 (`2_360`) por `3_000` — simula un desarrollo mucho peor de lo esperado en ese año — y vuelve a ejecutar todo el flujo (factores, proyección, provisión). ¿Cómo cambia el factor de desarrollo del periodo 3 al 4? ¿Y la provisión total?

**Para ti:** el estándar Chain Ladder asume que los factores de desarrollo son estables en el tiempo (no dependen del año de origen). Añade una función `factores_por_diagonal(triangulo)` que calcule, para cada año de origen individualmente, el ratio observado entre columnas consecutivas (sin agregarlos todos), y compáralos con los factores volumen-ponderados de la sección 2 — ¿hay algún año de origen que se desvíe bastante del resto? Eso, en la práctica, sería una señal para investigar (cambio en el proceso de tramitación, un siniestro grande atípico...) antes de aplicar el método sin más.

## Resumen

- Un triángulo de siniestralidad acumulada se representa de forma natural como `DataFrame` de pandas: años de origen en el índice, periodos de desarrollo en las columnas, `NaN` en el futuro no observado.
- Los factores de desarrollo (*link ratios*) volumen-ponderados se calculan como la suma de la columna siguiente entre la suma de la columna actual, usando solo los años de origen con dato en ambas columnas — no como la media simple de ratios individuales.
- Proyectar el triángulo es aplicar esos factores sucesivamente a la última celda observada de cada fila, hasta completar el rectángulo (*ultimate*).
- La provisión de cada año de origen es `ultimate - pagado hasta hoy`; la provisión total es la suma por años de origen.
- Todo cálculo de provisiones debe validarse con casos de control: un año totalmente desarrollado debe dar provisión cero, y un triángulo sintético con un patrón de desarrollo conocido debe reproducirse exactamente — el mismo principio de "valida siempre numéricamente" de los notebooks anteriores de esta unidad (Black-Scholes vs. Monte Carlo, cópula con $\rho=0$).
- Como con cualquier cálculo actuarial implementado con ayuda de una IA: el método, los supuestos (estabilidad de los factores de desarrollo, tratamiento de años atípicos) y la validación del resultado siguen siendo responsabilidad profesional vuestra — la IA puede acelerar la escritura del código de `calcular_factores_desarrollo`/`proyectar_triangulo` una vez decidido el método, no decidir si Chain Ladder es el método adecuado para vuestra cartera.

**Con esto se cierra el bloque de teoría del Día 10.** Los ejercicios (`08_ejercicios.ipynb`/`09_soluciones.ipynb`) combinan práctica de ML aplicado (Unidad 11) y de cálculo actuarial-financiero (Unidad 12). El cierre del día — el proyecto final opcional — vive en `proyecto_final/`, en la raíz del repositorio, no en esta carpeta: ver el `README.md` de este directorio para el enlace.